In [5]:
from datasets import load_dataset

dataset = load_dataset("ag_news")

In [6]:
import pandas as pd
import numpy as numpy
from google.colab import drive


In [7]:
dataset["train"][1]

{'text': 'Carlyle Looks Toward Commercial Aerospace (Reuters) Reuters - Private investment firm Carlyle Group,\\which has a reputation for making well-timed and occasionally\\controversial plays in the defense industry, has quietly placed\\its bets on another part of the market.',
 'label': 2}

In [8]:
print(dataset['train']['text'][:20])

["Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\\band of ultra-cynics, are seeing green again.", 'Carlyle Looks Toward Commercial Aerospace (Reuters) Reuters - Private investment firm Carlyle Group,\\which has a reputation for making well-timed and occasionally\\controversial plays in the defense industry, has quietly placed\\its bets on another part of the market.', "Oil and Economy Cloud Stocks' Outlook (Reuters) Reuters - Soaring crude prices plus worries\\about the economy and the outlook for earnings are expected to\\hang over the stock market next week during the depth of the\\summer doldrums.", 'Iraq Halts Oil Exports from Main Southern Pipeline (Reuters) Reuters - Authorities have halted oil export\\flows from the main pipeline in southern Iraq after\\intelligence showed a rebel militia could strike\\infrastructure, an oil official said on Saturday.', 'Oil prices soar to all-time record, posing new menace to US economy (AFP) 

In [9]:
# Filter for just the 'Sports' category (Label 1 in AG News)
sports_news = dataset['train'].filter(lambda x: x['label'] == 1)

# Function to add a 'start' and 'end' token so the model knows where news begins/ends
def format_for_generation(example):
    return {"text": f"<|startoftext|> {example['text']} <|endoftext|>"}

gen_dataset = sports_news.map(format_for_generation)

In [10]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token # GPT-2 doesn't have a pad token by default

def tokenize_fn(examples):
    return tokenizer(examples["text"], truncation=True, max_length=128, padding="max_length")

tokenized_gen = gen_dataset.map(tokenize_fn, batched=True)

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/30000 [00:00<?, ? examples/s]

In [11]:
from transformers import AutoModelForCausalLM, TrainingArguments, Trainer

model = AutoModelForCausalLM.from_pretrained("gpt2")

training_args = TrainingArguments(
    output_dir="./news_gen_model",
    per_device_train_batch_size=8,
    num_train_epochs=3,
    save_steps=500,
    logging_steps=100,
    evaluation_strategy="no" # Generation is often evaluated via Perplexity later
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_gen,
    # Data collator for language modeling automatically handles shifting the labels
    data_collator=None 
)

trainer.train()

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

TypeError: TrainingArguments.__init__() got an unexpected keyword argument 'evaluation_strategy'

In [ ]:
from transformers import pipeline

gen_pipe = pipeline("text-generation", model="./news_gen_model", tokenizer="gpt2")

# Generate 5 examples
results = gen_pipe("<|startoftext|>", max_length=100, num_return_sequences=5)

for i, res in enumerate(results):
    print(f"Synthetic Sports News {i}: {res['generated_text']}")

OSError: Repo id must use alphanumeric chars, '-', '_' or '.'. The name cannot start or end with '-' or '.' and the maximum length is 96: './news_gen_model'.

In [ ]:
import torch

# Check if a GPU is available
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Move your model to the GPU
model.to(device)

# When tokenizing, the Trainer API usually handles moving data for you,
# but if you do it manually, move your tensors:
# inputs = inputs.to(device)

Using device: cuda


NameError: name 'model' is not defined